In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, LongType, DateType, TimestampType, DoubleType
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, date_format, round, lag, when, avg, lit, to_date, abs
from pyspark.sql import Window
import numpy as np

In [0]:
jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Historical_Prices",
    properties=connection_properties
)

In [0]:
old_data

In [0]:
expected_schema = StructType([
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", IntegerType(), True),
    StructField("Dividends", DoubleType(), True),
    StructField("Stock_Symbol", StringType(), True),
    StructField("Stock_Splits", DoubleType(), True),
    StructField("Date", TimestampType(), True)
])

double_type_columns = ["Open", "High", "Low", "Close"]

for field in expected_schema:
    if field.name not in old_data.columns:
        raise ValueError(f"Column {field.name} is missing in the old data")

for field in expected_schema:
    if field.dataType == TimestampType():
        old_data = old_data.withColumn(field.name, to_timestamp(col(field.name)))
    else:
        old_data = old_data.withColumn(field.name, col(field.name).cast(field.dataType))

for column in double_type_columns:
    old_data = old_data.withColumn(column, round(col(column), 2).cast(DoubleType()))

old_data = old_data.withColumn("Date", to_timestamp("Date"))
old_data = old_data.withColumn("Date", from_utc_timestamp("Date", "America/New_York"))
old_data = old_data.withColumn("Date", date_format("Date", "yyyy-MM-dd HH:mm:ss"))

In [0]:
new_data = old_data.dropDuplicates()
processed_data = new_data.dropna(subset=["Date", "Open", "High", "Low", "Close", "Volume", "Stock_Symbol"])

In [0]:
processed_data.write.mode("overwrite").parquet("/dbfs/FileStore/Silver/Historical_Prices_Silver.parquet")

In [0]:
# IF NOT EXISTS (
#     SELECT * FROM INFORMATION_SCHEMA.TABLES 
#     WHERE TABLE_NAME = 'Historical_Prices' AND TABLE_SCHEMA = 'Silver'
# )
# BEGIN
#     CREATE TABLE Silver.Historical_Prices (
#         [Open] FLOAT NOT NULL,
#         [High] FLOAT NOT NULL,
#         [Low] FLOAT NOT NULL,
#         [Close] FLOAT NOT NULL,
#         [Volume] INT NOT NULL,
#         [Dividends] FLOAT NULL,
#         [Stock_Symbol] NVARCHAR(50) NOT NULL,
#         [Stock_Splits] FLOAT NULL,
#         [Date] DATETIME NOT NULL
#     );
# END


In [0]:
df_spark_historical_prices_silver = spark.read.parquet(
    "/dbfs/FileStore/Silver/Historical_Prices_Silver.parquet",
    header=True,
    inferSchema=True
)

In [0]:
df_spark_historical_prices_silver.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Historical_Prices") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")

In [0]:
df_spark_historical_prices_silver.printSchema()

In [0]:
def calculate_rsi_numpy(close_prices, period=14):
    delta = np.diff(close_prices)
    delta = np.concatenate(([np.nan], delta))

    gains = np.where(delta > 0, delta, 0)
    losses = np.where(delta < 0, -delta, 0)

    avg_gains = np.zeros_like(close_prices)
    avg_losses = np.zeros_like(close_prices)
    rsi = np.zeros_like(close_prices, dtype=float)

    if len(close_prices) >= period:
        avg_gains[period-1] = np.mean(gains[1:period+1])
        avg_losses[period-1] = np.mean(losses[1:period+1])
    
    for i in range(period, len(close_prices)):
        avg_gains[i] = (avg_gains[i-1] * (period - 1) + gains[i]) / period
        avg_losses[i] = (avg_losses[i-1] * (period - 1) + losses[i]) / period
    
    with np.errstate(divide='ignore', invalid='ignore'):
        rs = avg_gains / avg_losses
        rsi = np.where(avg_losses != 0, 100 - (100 / (1 + rs)), 100)
    
    rsi[:period-1] = np.nan
    return np.round(rsi, 2).tolist(), np.round(gains, 2).tolist(), np.round(losses, 2).tolist()

periods = list(range(2, 30))

df_old_data = df_spark_historical_prices_silver.orderBy("Stock_Symbol", "Date").select("Stock_Symbol", "Date", "Close")

for period in periods:
    windowSpec = Window.partitionBy("Stock_Symbol").orderBy("Date")
    df_old_data = df_old_data.withColumn(f"delta_{period}",   \
                        col("Close") - lag("Close", 1)  \
                        .over(windowSpec))

    df_old_data = df_old_data.withColumn(f"gain_{period}",
                        when(col(f"delta_{period}") > 0, col(f"delta_{period}"))    \
                        .otherwise(0))
    
    df_old_data = df_old_data.withColumn(f"loss_{period}", when(col(f"delta_{period}") < 0, -col(f"delta_{period}"))  \
                       .otherwise(0))

    avg_window = Window.partitionBy("Stock_Symbol") \
                        .orderBy("Date")    \
                        .rowsBetween(-period + 1, 0)

    df_old_data = df_old_data.withColumn(f"avg_gain_{period}",    \
                        avg(col(f"gain_{period}"))  \
                        .over(avg_window))  
    
    df_old_data = df_old_data.withColumn(f"avg_loss_{period}",    \
                        avg(col(f"loss_{period}"))  \
                        .over(avg_window))  

    df_old_data = df_old_data.withColumn(f"RS_{period}", when(col(f"avg_loss_{period}") == 0, lit(None))  \
                        .otherwise(col(f"avg_gain_{period}") / col(f"avg_loss_{period}")))
    
    df_old_data = df_old_data.withColumn(f"RSI_{period}",
                        when(col(f"RS_{period}").isNotNull(),   \
                              100 - (100 / (1 + col(f"RS_{period}")))   \
                              ))

selected_cols = ["Stock_Symbol", "Date", "Close"] + \
    [f"RSI_{p}" for p in periods] + \
    [f"gain_{p}" for p in periods] + \
    [f"loss_{p}" for p in periods]

df_result_ta = df_old_data.select(*selected_cols)

for period in periods:
    df_result_ta = df_result_ta.withColumn(f"RSI_{period}", round(col(f"RSI_{period}"), 2))
    df_result_ta = df_result_ta.withColumn(f"gain_{period}", round(col(f"gain_{period}"), 2))
    df_result_ta = df_result_ta.withColumn(f"loss_{period}", round(col(f"loss_{period}"), 2))

for period in periods:
    ma_window = Window.partitionBy("Stock_Symbol").orderBy("Date").rowsBetween(-period + 1, 0)

    df_result_ta = df_result_ta.withColumn(f"rolling_avg_{period}", \
                                    avg(col("Close")).over(ma_window))

    df_result_ta = df_result_ta.withColumn(f"abs_diff_{period}",    \
                                    abs(col("Close") - col(f"rolling_avg_{period}")))

    df_result_ta = df_result_ta.withColumn(f"rel_diff_{period}",    \
                                    when(col("Close") != 0, \
                                         col(f"abs_diff_{period}") / col("Close"))  \
                                    .otherwise(lit(None)))

    df_result_ta = df_result_ta.withColumn(f"MARD_{period}",    \
                                    avg(col(f"rel_diff_{period}")).over(ma_window))
    
    df_result_ta = df_result_ta.withColumn(f"MARD_{period}", round(col(f"MARD_{period}"), 4))
    df_result_ta = df_result_ta.drop(f"rolling_avg_{period}", f"abs_diff_{period}", f"rel_diff_{period}")

df_result_ta = df_result_ta.fillna(-99999)

In [0]:
df_result_ta = df_result_ta.withColumn("Date", date_format("Date", "yyyy-MM-dd HH:mm:ss"))
df_result_ta = df_result_ta.withColumn("Date", to_date("Date"))

In [0]:
df_result_ta.printSchema()

In [0]:
df_result_ta.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Silver.Historical_Prices_TA") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")

In [0]:
# IF NOT EXISTS (
#     SELECT * FROM INFORMATION_SCHEMA.TABLES 
#     WHERE TABLE_NAME = 'Historical_Prices_TA' AND TABLE_SCHEMA = 'Silver'
# )
# BEGIN
# CREATE TABLE Silver.Historical_Prices_TA (
#     [Stock_Symbol] NVARCHAR(50) NOT NULL,
#     [Date] DATETIME NOT NULL,
#     [Close] FLOAT NOT NULL,
#     [RSI_2] FLOAT NOT NULL,
#     [RSI_3] FLOAT NOT NULL,
#     [RSI_4] FLOAT NOT NULL,
#     [RSI_5] FLOAT NOT NULL,
#     [RSI_6] FLOAT NOT NULL,
#     [RSI_7] FLOAT NOT NULL,
#     [RSI_8] FLOAT NOT NULL,
#     [RSI_9] FLOAT NOT NULL,
#     [RSI_10] FLOAT NOT NULL,
#     [RSI_11] FLOAT NOT NULL,
#     [RSI_12] FLOAT NOT NULL,
#     [RSI_13] FLOAT NOT NULL,
#     [RSI_14] FLOAT NOT NULL,
#     [RSI_15] FLOAT NOT NULL,
#     [RSI_16] FLOAT NOT NULL,
#     [RSI_17] FLOAT NOT NULL,
#     [RSI_18] FLOAT NOT NULL,
#     [RSI_19] FLOAT NOT NULL,
#     [RSI_20] FLOAT NOT NULL,
#     [RSI_21] FLOAT NOT NULL,
#     [RSI_22] FLOAT NOT NULL,
#     [RSI_23] FLOAT NOT NULL,
#     [RSI_24] FLOAT NOT NULL,
#     [RSI_25] FLOAT NOT NULL,
#     [RSI_26] FLOAT NOT NULL,
#     [RSI_27] FLOAT NOT NULL,
#     [RSI_28] FLOAT NOT NULL,
#     [RSI_29] FLOAT NOT NULL,
#     [gain_2] FLOAT NOT NULL,
#     [loss_2] FLOAT NOT NULL,
#     [gain_3] FLOAT NOT NULL,
#     [loss_3] FLOAT NOT NULL,
#     [gain_4] FLOAT NOT NULL,
#     [loss_4] FLOAT NOT NULL,
#     [gain_5] FLOAT NOT NULL,
#     [loss_5] FLOAT NOT NULL,
#     [gain_6] FLOAT NOT NULL,
#     [loss_6] FLOAT NOT NULL,
#     [gain_7] FLOAT NOT NULL,
#     [loss_7] FLOAT NOT NULL,
#     [gain_8] FLOAT NOT NULL,
#     [loss_8] FLOAT NOT NULL,
#     [gain_9] FLOAT NOT NULL,
#     [loss_9] FLOAT NOT NULL,
#     [gain_10] FLOAT NOT NULL,
#     [loss_10] FLOAT NOT NULL,
#     [gain_11] FLOAT NOT NULL,
#     [loss_11] FLOAT NOT NULL,
#     [gain_12] FLOAT NOT NULL,
#     [loss_12] FLOAT NOT NULL,
#     [gain_13] FLOAT NOT NULL,
#     [loss_13] FLOAT NOT NULL,
#     [gain_14] FLOAT NOT NULL,
#     [loss_14] FLOAT NOT NULL,
#     [gain_15] FLOAT NOT NULL,
#     [loss_15] FLOAT NOT NULL,
#     [gain_16] FLOAT NOT NULL,
#     [loss_16] FLOAT NOT NULL,
#     [gain_17] FLOAT NOT NULL,
#     [loss_17] FLOAT NOT NULL,
#     [gain_18] FLOAT NOT NULL,
#     [loss_18] FLOAT NOT NULL,
#     [gain_19] FLOAT NOT NULL,
#     [loss_19] FLOAT NOT NULL,
#     [gain_20] FLOAT NOT NULL,
#     [loss_20] FLOAT NOT NULL,
#     [gain_21] FLOAT NOT NULL,
#     [loss_21] FLOAT NOT NULL,
#     [gain_22] FLOAT NOT NULL,
#     [loss_22] FLOAT NOT NULL,
#     [gain_23] FLOAT NOT NULL,
#     [loss_23] FLOAT NOT NULL,
#     [gain_24] FLOAT NOT NULL,
#     [loss_24] FLOAT NOT NULL,
#     [gain_25] FLOAT NOT NULL,
#     [loss_25] FLOAT NOT NULL,
#     [gain_26] FLOAT NOT NULL,
#     [loss_26] FLOAT NOT NULL,
#     [gain_27] FLOAT NOT NULL,
#     [loss_27] FLOAT NOT NULL,
#     [gain_28] FLOAT NOT NULL,
#     [loss_28] FLOAT NOT NULL,
#     [gain_29] FLOAT NOT NULL,
#     [loss_29] FLOAT NOT NULL,
#     [MARD_2] FLOAT NOT NULL,
#     [MARD_3] FLOAT NOT NULL,
#     [MARD_4] FLOAT NOT NULL,
#     [MARD_5] FLOAT NOT NULL,
#     [MARD_6] FLOAT NOT NULL,
#     [MARD_7] FLOAT NOT NULL,
#     [MARD_8] FLOAT NOT NULL,
#     [MARD_9] FLOAT NOT NULL,
#     [MARD_10] FLOAT NOT NULL,
#     [MARD_11] FLOAT NOT NULL,
#     [MARD_12] FLOAT NOT NULL,
#     [MARD_13] FLOAT NOT NULL,
#     [MARD_14] FLOAT NOT NULL,
#     [MARD_15] FLOAT NOT NULL,
#     [MARD_16] FLOAT NOT NULL,
#     [MARD_17] FLOAT NOT NULL,
#     [MARD_18] FLOAT NOT NULL,
#     [MARD_19] FLOAT NOT NULL,
#     [MARD_20] FLOAT NOT NULL,
#     [MARD_21] FLOAT NOT NULL,
#     [MARD_22] FLOAT NOT NULL,
#     [MARD_23] FLOAT NOT NULL,
#     [MARD_24] FLOAT NOT NULL,
#     [MARD_25] FLOAT NOT NULL,
#     [MARD_26] FLOAT NOT NULL,
#     [MARD_27] FLOAT NOT NULL,
#     [MARD_28] FLOAT NOT NULL,
#     [MARD_29] FLOAT NOT NULL
# );
# END